# Water Distribution System — Complete Signal Analysis
## Wave Propagation Characterization Across Network Topologies
### Aghashahi et al. (2022) Dataset | MSSP/RESS Research

**Research Focus:** Cross-sensor wave propagation analysis — characterizing leak signatures through
time-frequency coherence and inter-sensor delay estimation across network topologies.

---
**Dataset:** `E:\Upwork Project\AI_Leak_Detection_Project\data\raw\`  
**Sensors:** Accelerometer (A1, A2) | Dynamic Pressure (P1, P2) | Hydrophone (H1, H2)  
**Topologies:** Branched (BR) | Looped (LO)  
**Leak Types:** CC | LC | GL | OL | NL  
**Flow Conditions:** 0.18 LPS | 0.47 LPS | ND | Transient  
---

## 0. Imports & Configuration

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns
from scipy import signal as sp_signal
from scipy.stats import kurtosis, skewness, entropy
from scipy.fft import fft, fftfreq
from scipy.signal import correlate, correlation_lags, welch, coherence, hilbert
import pywt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import mutual_info_score
from tqdm import tqdm
import json

warnings.filterwarnings('ignore')

# ─── PATHS ────────────────────────────────────────────────────────────────────
BASE_DIR = r"E:\Upwork Project\AI_Leak_Detection_Project\data\raw"
ACC_DIR  = os.path.join(BASE_DIR, "Accelerometer")
DYN_DIR  = os.path.join(BASE_DIR, "Dynamic Pressure Sensor")
HYD_DIR  = os.path.join(BASE_DIR, "Hydrophone")
OUT_DIR  = r"E:\Upwork Project\AI_Leak_Detection_Project\results\figures"
os.makedirs(OUT_DIR, exist_ok=True)

# ─── DATASET CONSTANTS ────────────────────────────────────────────────────────
FS_ACC  = 25600   # Hz — accelerometer & dynamic pressure (measured from data)
FS_HYD  = 8000    # Hz — hydrophone

LEAK_TYPES  = ['Circumferential Crack', 'Longitudinal Crack',
               'Gasket Leak', 'Orifice Leak', 'No-leak']
LEAK_CODES  = {'Circumferential Crack': 'CC', 'Longitudinal Crack': 'LC',
               'Gasket Leak': 'GL', 'Orifice Leak': 'OL', 'No-leak': 'NL'}
TOPOLOGIES  = ['Branched', 'Looped']
TOPO_CODES  = {'Branched': 'BR', 'Looped': 'LO'}
FLOW_CONDS  = ['0.18 LPS', '0.47 LPS', 'ND', 'Transient']

# ─── COLORS ───────────────────────────────────────────────────────────────────
LEAK_COLORS = {
    'CC': '#E63946', 'LC': '#457B9D', 'GL': '#2A9D8F',
    'OL': '#E9C46A', 'NL': '#6D6875'
}
TOPO_COLORS = {'Branched': '#E63946', 'Looped': '#457B9D'}
FLOW_COLORS = {'0.18 LPS': '#2A9D8F', '0.47 LPS': '#E9C46A',
               'ND': '#457B9D', 'Transient': '#E63946'}

# ─── MATPLOTLIB STYLE ─────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.facecolor': 'white',
    'axes.facecolor': '#F8F9FA',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'savefig.bbox_inches': 'tight',
    'savefig.facecolor': 'white'
})

print('✅ All imports successful')
print(f'📁 Output directory: {OUT_DIR}')
print(f'📊 Accelerometer Fs: {FS_ACC} Hz')
print(f'📊 Hydrophone Fs:    {FS_HYD} Hz')

## 1. Data Loading Utilities

In [ ]:
def load_acc_csv(filepath):
    """Load accelerometer or dynamic pressure CSV. Returns (time_array, value_array)."""
    df = pd.read_csv(filepath)
    return df['Sample'].values, df['Value'].values


def load_hydrophone_raw(filepath, fs=8000, duration=30):
    """Load hydrophone .raw binary file. Returns (time_array, value_array)."""
    n_samples = fs * duration
    data = np.fromfile(filepath, dtype=np.float64)
    if len(data) > n_samples:
        data = data[:n_samples]
    t = np.arange(len(data)) / fs
    return t, data


def find_file(sensor_dir, topology, leak_type, flow, sensor_id):
    """
    Find a file matching topology/leak_type/flow/sensor_id pattern.
    sensor_id: e.g. 'A1', 'A2', 'H1', 'H2', 'P1', 'P2'
    """
    folder = os.path.join(sensor_dir, topology, leak_type)
    # Build partial name from flow condition
    flow_map = {'0.18 LPS': '0.18', '0.47 LPS': '0.47', 'ND': 'ND', 'Transient': 'Transient'}
    flow_key = flow_map.get(flow, flow)
    pattern = os.path.join(folder, f"*{flow_key}*{sensor_id}*")
    matches = glob.glob(pattern)
    if not matches:
        return None
    return matches[0]


def load_sensor_pair(sensor_dir, topology, leak_type, flow,
                     s1='A1', s2='A2', fs=FS_ACC, is_raw=False):
    """
    Load paired sensors for a given condition.
    Returns (t1, sig1, t2, sig2) or None if files missing.
    """
    f1 = find_file(sensor_dir, topology, leak_type, flow, s1)
    f2 = find_file(sensor_dir, topology, leak_type, flow, s2)
    if f1 is None or f2 is None:
        return None
    if is_raw:
        t1, sig1 = load_hydrophone_raw(f1, fs=fs)
        t2, sig2 = load_hydrophone_raw(f2, fs=fs)
    else:
        t1, sig1 = load_acc_csv(f1)
        t2, sig2 = load_acc_csv(f2)
    # Trim to same length
    n = min(len(sig1), len(sig2))
    return t1[:n], sig1[:n], t2[:n], sig2[:n]


print('✅ Data loading utilities ready')

# ─── Quick sanity check on uploaded file ──────────────────────────────────────
# Adjust path to your local file:
test_file = os.path.join(ACC_DIR, 'Branched', 'Circumferential Crack',
                         'BR_CC_0.18 LPS_A1.csv')
t_test, v_test = load_acc_csv(test_file)
print(f'  Samples: {len(v_test):,}')
print(f'  Duration: {t_test[-1]:.1f} s')
print(f'  Fs (measured): {1/(t_test[1]-t_test[0]):.0f} Hz')
print(f'  Value range: [{v_test.min():.4f}, {v_test.max():.4f}]')

## 2. SECTION A — Time-Domain Signal Characterization
### Figure 1: Raw Signal Overview — All Leak Types (A1 sensor, 0.18 LPS, Branched)

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(16, 14), sharex=False)
fig.suptitle('Time-Domain Accelerometer Signals — All Leak Types\n'
             'Branched Network | Flow: 0.18 LPS | Sensor A1', fontsize=14, fontweight='bold')

for idx, leak in enumerate(LEAK_TYPES):
    code = LEAK_CODES[leak]
    fpath = find_file(ACC_DIR, 'Branched', leak, '0.18 LPS', 'A1')
    if fpath is None:
        axes[idx].text(0.5, 0.5, f'File not found: {leak}', ha='center', transform=axes[idx].transAxes)
        continue
    t, v = load_acc_csv(fpath)
    # Plot first 5 seconds
    mask = t <= 5
    axes[idx].plot(t[mask], v[mask], color=LEAK_COLORS[code], lw=0.6, alpha=0.9)
    axes[idx].set_ylabel('Acceleration\n[g]', fontsize=10)
    axes[idx].set_title(f'{leak} ({code})', fontsize=11, fontweight='bold',
                        color=LEAK_COLORS[code])
    axes[idx].set_xlim(0, 5)
    # Annotate RMS
    rms = np.sqrt(np.mean(v**2))
    axes[idx].annotate(f'RMS={rms:.5f} g', xy=(4.5, v[mask].max()*0.85),
                       fontsize=9, color='black',
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

axes[-1].set_xlabel('Time [s]', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig01_TimeDomain_AllLeakTypes.png'))
plt.show()
print('✅ Figure 1 saved')

### Figure 2: Sensor Pair Comparison — A1 vs A2 (Same Condition)

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(18, 14))
fig.suptitle('Inter-Sensor Comparison: A1 vs A2\nBranched Network | Flow: 0.18 LPS',
             fontsize=14, fontweight='bold')

for idx, leak in enumerate(LEAK_TYPES):
    code = LEAK_CODES[leak]
    result = load_sensor_pair(ACC_DIR, 'Branched', leak, '0.18 LPS', 'A1', 'A2')
    if result is None:
        continue
    t1, sig1, t2, sig2 = result
    mask = t1 <= 3

    axes[idx, 0].plot(t1[mask], sig1[mask], color=LEAK_COLORS[code], lw=0.6)
    axes[idx, 0].set_title(f'{code} — A1', fontsize=10, fontweight='bold')
    axes[idx, 0].set_ylabel('Accel [g]')

    axes[idx, 1].plot(t2[mask], sig2[mask], color=LEAK_COLORS[code], lw=0.6, alpha=0.8)
    axes[idx, 1].set_title(f'{code} — A2', fontsize=10, fontweight='bold')

for ax in axes[-1]:
    ax.set_xlabel('Time [s]')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig02_SensorPair_A1vsA2.png'))
plt.show()
print('✅ Figure 2 saved')

### Figure 3: Flow Condition Effect on Signal Morphology

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=False)
fig.suptitle('Effect of Flow Condition on Accelerometer Signal\n'
             'Branched | Circumferential Crack | Sensor A1', fontsize=14, fontweight='bold')

for idx, flow in enumerate(FLOW_CONDS):
    fpath = find_file(ACC_DIR, 'Branched', 'Circumferential Crack', flow, 'A1')
    if fpath is None:
        continue
    t, v = load_acc_csv(fpath)
    mask = t <= 5
    axes[idx].plot(t[mask], v[mask], color=FLOW_COLORS[flow], lw=0.7)
    axes[idx].set_ylabel('Accel [g]')
    axes[idx].set_title(f'Flow: {flow}', fontsize=11, color=FLOW_COLORS[flow], fontweight='bold')
    rms = np.sqrt(np.mean(v**2))
    kurt = kurtosis(v)
    axes[idx].annotate(f'RMS={rms:.5f} | Kurt={kurt:.2f}',
                       xy=(0.02, 0.88), xycoords='axes fraction', fontsize=9,
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

axes[-1].set_xlabel('Time [s]')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig03_FlowConditionEffect.png'))
plt.show()
print('✅ Figure 3 saved')

### Figure 4: Topology Comparison — Branched vs Looped (Same Leak, Same Flow)

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(18, 14))
fig.suptitle('Network Topology Effect on Acoustic Signatures\nBranched vs Looped | Flow: 0.18 LPS | A1',
             fontsize=14, fontweight='bold')

for idx, leak in enumerate(LEAK_TYPES):
    code = LEAK_CODES[leak]
    for jdx, topo in enumerate(TOPOLOGIES):
        fpath = find_file(ACC_DIR, topo, leak, '0.18 LPS', 'A1')
        if fpath is None:
            axes[idx, jdx].text(0.5, 0.5, 'Not found', ha='center', transform=axes[idx, jdx].transAxes)
            continue
        t, v = load_acc_csv(fpath)
        mask = t <= 3
        axes[idx, jdx].plot(t[mask], v[mask], color=TOPO_COLORS[topo], lw=0.6)
        axes[idx, jdx].set_title(f'{code} — {topo}', fontsize=10,
                                  color=TOPO_COLORS[topo], fontweight='bold')
        axes[idx, jdx].set_ylabel('Accel [g]')

for ax in axes[-1]:
    ax.set_xlabel('Time [s]')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig04_TopologyComparison.png'))
plt.show()
print('✅ Figure 4 saved')

## 3. SECTION B — Frequency Domain Analysis
### Figure 5: Power Spectral Density — All Leak Types

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Power Spectral Density — All Leak Types\nBranched | 0.18 LPS', fontsize=14, fontweight='bold')

for topo_idx, topo in enumerate(TOPOLOGIES):
    for leak in LEAK_TYPES:
        code = LEAK_CODES[leak]
        fpath = find_file(ACC_DIR, topo, leak, '0.18 LPS', 'A1')
        if fpath is None:
            continue
        t, v = load_acc_csv(fpath)
        freqs, psd = welch(v, fs=FS_ACC, nperseg=2048, noverlap=1024)
        axes[topo_idx].semilogy(freqs, psd, label=code, color=LEAK_COLORS[code], lw=1.5, alpha=0.85)

    axes[topo_idx].set_xlabel('Frequency [Hz]')
    axes[topo_idx].set_ylabel('PSD [g²/Hz]')
    axes[topo_idx].set_title(f'{topo} Network')
    axes[topo_idx].legend()
    axes[topo_idx].set_xlim(0, FS_ACC/2)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig05_PSD_AllLeakTypes.png'))
plt.show()
print('✅ Figure 5 saved')

### Figure 6: FFT Magnitude Spectrum — Leak vs No-Leak

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('FFT Spectrum: Leak vs No-Leak Comparison\nAll Flow Conditions | Branched | A1',
             fontsize=14, fontweight='bold')

for idx, flow in enumerate(FLOW_CONDS):
    ax = axes[idx // 2, idx % 2]
    for leak in ['Circumferential Crack', 'No-leak']:
        code = LEAK_CODES[leak]
        fpath = find_file(ACC_DIR, 'Branched', leak, flow, 'A1')
        if fpath is None:
            continue
        t, v = load_acc_csv(fpath)
        N = len(v)
        yf = np.abs(fft(v - v.mean()))[:N//2]
        xf = fftfreq(N, 1/FS_ACC)[:N//2]
        ax.semilogy(xf, yf, color=LEAK_COLORS[code], lw=0.8,
                    label=f'{code}', alpha=0.85)

    ax.set_title(f'Flow: {flow}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Frequency [Hz]')
    ax.set_ylabel('Magnitude')
    ax.legend()
    ax.set_xlim(0, 5000)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig06_FFT_LeakVsNoLeak.png'))
plt.show()
print('✅ Figure 6 saved')

### Figure 7: Frequency Band Energy Distribution

In [ ]:
def band_energy(signal, fs, f_low, f_high):
    freqs, psd = welch(signal, fs=fs, nperseg=1024)
    mask = (freqs >= f_low) & (freqs <= f_high)
    return np.trapz(psd[mask], freqs[mask])

BANDS = [
    ('Sub-100Hz',    0,    100),
    ('100-500Hz',  100,    500),
    ('500-2kHz',   500,   2000),
    ('2k-5kHz',   2000,   5000),
    ('5k-12kHz',  5000,  12000),
    ('>12kHz',   12000,  FS_ACC//2),
]

energy_data = {}
for leak in LEAK_TYPES:
    code = LEAK_CODES[leak]
    fpath = find_file(ACC_DIR, 'Branched', leak, '0.18 LPS', 'A1')
    if fpath is None:
        continue
    t, v = load_acc_csv(fpath)
    energies = [band_energy(v, FS_ACC, fl, fh) for _, fl, fh in BANDS]
    total = sum(energies)
    energy_data[code] = [e/total*100 for e in energies]

band_labels = [b[0] for b in BANDS]
df_energy = pd.DataFrame(energy_data, index=band_labels)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Frequency Band Energy Distribution\nBranched | 0.18 LPS | A1',
             fontsize=14, fontweight='bold')

# Grouped bar chart
x = np.arange(len(band_labels))
width = 0.15
for i, (code, vals) in enumerate(energy_data.items()):
    ax1.bar(x + i*width, vals, width, label=code,
            color=LEAK_COLORS[code], alpha=0.85, edgecolor='white')
ax1.set_xticks(x + width*2)
ax1.set_xticklabels(band_labels, rotation=30, ha='right')
ax1.set_ylabel('Energy Fraction [%]')
ax1.set_title('Band Energy per Leak Type')
ax1.legend()

# Heatmap
sns.heatmap(df_energy, ax=ax2, annot=True, fmt='.1f', cmap='YlOrRd',
            cbar_kws={'label': 'Energy %'}, linewidths=0.5)
ax2.set_title('Energy Heatmap [%]')
ax2.set_xlabel('Leak Type')
ax2.set_ylabel('Frequency Band')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig07_BandEnergy.png'))
plt.show()
print('✅ Figure 7 saved')

## 4. SECTION C — Time-Frequency Analysis
### Figure 8: Short-Time Fourier Transform (STFT) Spectrograms

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(18, 18))
fig.suptitle('STFT Spectrograms — All Leak Types | Branched vs Looped | 0.18 LPS | A1',
             fontsize=14, fontweight='bold')

for idx, leak in enumerate(LEAK_TYPES):
    code = LEAK_CODES[leak]
    for jdx, topo in enumerate(TOPOLOGIES):
        ax = axes[idx, jdx]
        fpath = find_file(ACC_DIR, topo, leak, '0.18 LPS', 'A1')
        if fpath is None:
            continue
        t, v = load_acc_csv(fpath)
        # Use first 10 seconds
        n10 = int(10 * FS_ACC)
        v10 = v[:n10]
        f_stft, t_stft, Zxx = sp_signal.stft(v10, fs=FS_ACC, nperseg=512, noverlap=384)
        pcm = ax.pcolormesh(t_stft, f_stft, 20*np.log10(np.abs(Zxx)+1e-10),
                            shading='gouraud', cmap='inferno')
        ax.set_ylim(0, 5000)
        ax.set_title(f'{code} — {topo}', fontsize=10, fontweight='bold')
        ax.set_ylabel('Freq [Hz]')
        plt.colorbar(pcm, ax=ax, label='dB')

for ax in axes[-1]:
    ax.set_xlabel('Time [s]')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig08_STFT_Spectrograms.png'))
plt.show()
print('✅ Figure 8 saved')

### Figure 9: Continuous Wavelet Transform (CWT)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Continuous Wavelet Transform (Morlet) — Leak Type Comparison\nBranched | 0.18 LPS | A1',
             fontsize=14, fontweight='bold')

leaks_to_plot = LEAK_TYPES  # all 5
axes_flat = axes.flatten()

for idx, leak in enumerate(leaks_to_plot):
    code = LEAK_CODES[leak]
    fpath = find_file(ACC_DIR, 'Branched', leak, '0.18 LPS', 'A1')
    if fpath is None:
        continue
    t, v = load_acc_csv(fpath)
    # CWT on 2-second window
    n2 = int(2 * FS_ACC)
    v2 = v[:n2]
    scales = np.arange(1, 128)
    coeffs, freqs_cwt = pywt.cwt(v2, scales, 'morl', sampling_period=1/FS_ACC)
    t2 = np.linspace(0, 2, n2)

    im = axes_flat[idx].pcolormesh(t2, freqs_cwt, np.abs(coeffs),
                                    shading='auto', cmap='plasma')
    axes_flat[idx].set_ylim(0, 2000)
    axes_flat[idx].set_title(f'{leak} ({code})', fontsize=10,
                              color=LEAK_COLORS[code], fontweight='bold')
    axes_flat[idx].set_xlabel('Time [s]')
    axes_flat[idx].set_ylabel('Frequency [Hz]')
    plt.colorbar(im, ax=axes_flat[idx])

axes_flat[-1].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig09_CWT_AllLeaks.png'))
plt.show()
print('✅ Figure 9 saved')

### Figure 10: Wavelet Packet Decomposition Energy

In [ ]:
def wpd_energy(signal, wavelet='db4', level=4):
    wp = pywt.WaveletPacket(data=signal, wavelet=wavelet, mode='symmetric', maxlevel=level)
    nodes = [node.path for node in wp.get_level(level, 'freq')]
    energies = [np.sum(wp[node].data**2) for node in nodes]
    total = sum(energies) + 1e-12
    return np.array(energies) / total, nodes

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Wavelet Packet Decomposition Energy (db4, level 4)\nBranched | 0.18 LPS | A1',
             fontsize=14, fontweight='bold')

all_energies = {}
for leak in LEAK_TYPES:
    code = LEAK_CODES[leak]
    fpath = find_file(ACC_DIR, 'Branched', leak, '0.18 LPS', 'A1')
    if fpath is None:
        continue
    t, v = load_acc_csv(fpath)
    v_short = v[:int(5 * FS_ACC)]
    energies, nodes = wpd_energy(v_short)
    all_energies[code] = energies

df_wpd = pd.DataFrame(all_energies)
df_wpd.index = [f'N{i+1}' for i in range(len(df_wpd))]

# Bar chart
x = np.arange(len(df_wpd))
width = 0.15
for i, col in enumerate(df_wpd.columns):
    axes[0].bar(x + i*width, df_wpd[col].values*100, width,
                label=col, color=LEAK_COLORS[col], alpha=0.85)
axes[0].set_xlabel('Wavelet Packet Node')
axes[0].set_ylabel('Energy Fraction [%]')
axes[0].set_title('WPD Node Energy per Leak Type')
axes[0].legend()

# Heatmap
sns.heatmap(df_wpd.T * 100, ax=axes[1], cmap='Blues', annot=False,
            cbar_kws={'label': 'Energy %'}, linewidths=0.3)
axes[1].set_title('WPD Energy Heatmap')
axes[1].set_xlabel('Packet Node')
axes[1].set_ylabel('Leak Type')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig10_WPD_Energy.png'))
plt.show()
print('✅ Figure 10 saved')

## 5. SECTION D — Inter-Sensor Wave Propagation Analysis (CORE NOVEL CONTRIBUTION)
### Figure 11: Cross-Correlation — A1 vs A2 Inter-Arrival Delay

In [ ]:
def compute_gcc_phat(sig1, sig2, fs):
    """GCC-PHAT: Generalized Cross-Correlation with Phase Transform for TDOA."""
    N = len(sig1) + len(sig2) - 1
    N_fft = 2**int(np.ceil(np.log2(N)))
    S1 = np.fft.rfft(sig1, n=N_fft)
    S2 = np.fft.rfft(sig2, n=N_fft)
    G = S1 * np.conj(S2)
    G_phat = G / (np.abs(G) + 1e-10)
    cc = np.fft.irfft(G_phat, n=N_fft)
    lags = np.arange(-len(sig1)+1, len(sig2)) / fs
    cc_trim = np.concatenate([cc[-(len(sig1)-1):], cc[:len(sig2)]])
    return lags, cc_trim


fig, axes = plt.subplots(5, 2, figsize=(18, 16))
fig.suptitle('GCC-PHAT Cross-Correlation: A1 vs A2\nTDOA Estimation per Leak Type & Topology',
             fontsize=14, fontweight='bold')

tdoa_results = {}

for idx, leak in enumerate(LEAK_TYPES):
    code = LEAK_CODES[leak]
    tdoa_results[code] = {}
    for jdx, topo in enumerate(TOPOLOGIES):
        ax = axes[idx, jdx]
        result = load_sensor_pair(ACC_DIR, topo, leak, '0.18 LPS', 'A1', 'A2')
        if result is None:
            ax.text(0.5, 0.5, 'Files not found', ha='center', transform=ax.transAxes)
            continue
        t1, sig1, t2, sig2 = result
        # Use 5-second window
        n5 = int(5 * FS_ACC)
        s1, s2 = sig1[:n5], sig2[:n5]

        lags, cc = compute_gcc_phat(s1, s2, FS_ACC)
        # Focus on ±0.05 seconds window
        mask = np.abs(lags) <= 0.05
        peak_lag = lags[mask][np.argmax(np.abs(cc[mask]))]
        tdoa_results[code][topo] = peak_lag * 1000  # ms

        ax.plot(lags[mask]*1000, cc[mask], color=LEAK_COLORS[code], lw=1.2)
        ax.axvline(peak_lag*1000, color='red', ls='--', lw=1.5, alpha=0.8)
        ax.set_title(f'{code} — {topo} | TDOA={peak_lag*1000:.2f} ms',
                     fontsize=9, fontweight='bold')
        ax.set_ylabel('GCC-PHAT')

for ax in axes[-1]:
    ax.set_xlabel('Lag [ms]')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig11_GCCPHAT_TDOA.png'))
plt.show()
print('✅ Figure 11 saved')
print('\nTDOA Results [ms]:')
for code, topos in tdoa_results.items():
    print(f'  {code}: {topos}')

### Figure 12: TDOA Across All Flow Conditions & Topologies

In [ ]:
# Build TDOA matrix: leak × flow condition × topology
tdoa_matrix = {}

for topo in TOPOLOGIES:
    tdoa_matrix[topo] = {}
    for leak in LEAK_TYPES:
        code = LEAK_CODES[leak]
        tdoa_matrix[topo][code] = {}
        for flow in FLOW_CONDS:
            result = load_sensor_pair(ACC_DIR, topo, leak, flow, 'A1', 'A2')
            if result is None:
                tdoa_matrix[topo][code][flow] = np.nan
                continue
            t1, sig1, t2, sig2 = result
            n5 = int(5 * FS_ACC)
            s1, s2 = sig1[:n5], sig2[:n5]
            lags, cc = compute_gcc_phat(s1, s2, FS_ACC)
            mask = np.abs(lags) <= 0.05
            peak_lag = lags[mask][np.argmax(np.abs(cc[mask]))]
            tdoa_matrix[topo][code][flow] = peak_lag * 1000

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('TDOA (A1-A2) Across Flow Conditions & Leak Types\nGCC-PHAT Estimator',
             fontsize=14, fontweight='bold')

for idx, topo in enumerate(TOPOLOGIES):
    df_tdoa = pd.DataFrame(tdoa_matrix[topo]).T
    df_tdoa.columns = FLOW_CONDS
    sns.heatmap(df_tdoa, ax=axes[idx], annot=True, fmt='.2f',
                cmap='RdBu_r', center=0, linewidths=0.5,
                cbar_kws={'label': 'TDOA [ms]'})
    axes[idx].set_title(f'{topo} Network', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Flow Condition')
    axes[idx].set_ylabel('Leak Type')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig12_TDOA_HeatMap.png'))
plt.show()
print('✅ Figure 12 saved')

### Figure 13: Coherence Function Between A1 and A2

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Magnitude-Squared Coherence: A1 vs A2\nBranched | 0.18 LPS — All Leak Types',
             fontsize=14, fontweight='bold')
axes_flat = axes.flatten()

for idx, leak in enumerate(LEAK_TYPES):
    code = LEAK_CODES[leak]
    result = load_sensor_pair(ACC_DIR, 'Branched', leak, '0.18 LPS', 'A1', 'A2')
    if result is None:
        continue
    t1, sig1, t2, sig2 = result
    f_coh, Cxy = coherence(sig1, sig2, fs=FS_ACC, nperseg=2048)

    axes_flat[idx].plot(f_coh, Cxy, color=LEAK_COLORS[code], lw=1.2)
    axes_flat[idx].axhline(0.5, color='red', ls='--', lw=1, alpha=0.6, label='0.5 threshold')
    axes_flat[idx].fill_between(f_coh, 0, Cxy, alpha=0.2, color=LEAK_COLORS[code])
    axes_flat[idx].set_xlim(0, 5000)
    axes_flat[idx].set_ylim(0, 1.05)
    # Mean coherence
    mean_coh = np.mean(Cxy[(f_coh >= 0) & (f_coh <= 5000)])
    axes_flat[idx].set_title(f'{leak} ({code})\nMean Coh={mean_coh:.3f}',
                              fontsize=10, color=LEAK_COLORS[code], fontweight='bold')
    axes_flat[idx].set_xlabel('Frequency [Hz]')
    axes_flat[idx].set_ylabel('Coherence')
    axes_flat[idx].legend(fontsize=8)

axes_flat[-1].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig13_Coherence_A1A2.png'))
plt.show()
print('✅ Figure 13 saved')

### Figure 14: Topology Effect on Inter-Sensor Coherence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Topology Effect on Inter-Sensor Coherence (A1-A2)\nCircumferential Crack — All Flow Conditions',
             fontsize=14, fontweight='bold')

for jdx, topo in enumerate(TOPOLOGIES):
    for flow in FLOW_CONDS:
        result = load_sensor_pair(ACC_DIR, topo, 'Circumferential Crack', flow, 'A1', 'A2')
        if result is None:
            continue
        t1, sig1, t2, sig2 = result
        f_coh, Cxy = coherence(sig1, sig2, fs=FS_ACC, nperseg=2048)
        axes[jdx].plot(f_coh, Cxy, label=flow, color=FLOW_COLORS[flow], lw=1.5, alpha=0.85)

    axes[jdx].set_xlim(0, 5000)
    axes[jdx].set_ylim(0, 1.05)
    axes[jdx].set_title(f'{topo} Network', fontsize=12,
                         color=TOPO_COLORS[topo], fontweight='bold')
    axes[jdx].set_xlabel('Frequency [Hz]')
    axes[jdx].set_ylabel('Coherence (A1-A2)')
    axes[jdx].legend(title='Flow Condition')
    axes[jdx].axhline(0.5, color='black', ls=':', lw=1, alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig14_TopologyCoherence.png'))
plt.show()
print('✅ Figure 14 saved')

## 6. SECTION E — Envelope & Hilbert Analysis
### Figure 15: Hilbert Envelope — Leak Burst Detection

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(16, 14), sharex=False)
fig.suptitle('Hilbert Transform Envelope — Leak Signal Burst Analysis\n'
             'Branched | 0.18 LPS | A1 (First 10 seconds)',
             fontsize=14, fontweight='bold')

for idx, leak in enumerate(LEAK_TYPES):
    code = LEAK_CODES[leak]
    fpath = find_file(ACC_DIR, 'Branched', leak, '0.18 LPS', 'A1')
    if fpath is None:
        continue
    t, v = load_acc_csv(fpath)
    n10 = int(10 * FS_ACC)
    t10, v10 = t[:n10], v[:n10]

    analytic = hilbert(v10)
    envelope = np.abs(analytic)
    inst_freq = np.diff(np.unwrap(np.angle(analytic))) / (2*np.pi) * FS_ACC

    axes[idx].plot(t10, v10, color=LEAK_COLORS[code], lw=0.4, alpha=0.5, label='Signal')
    axes[idx].plot(t10, envelope, color='black', lw=1.2, label='Envelope')
    axes[idx].plot(t10, -envelope, color='black', lw=1.2)
    axes[idx].set_ylabel('Amplitude [g]')
    axes[idx].set_title(f'{leak} ({code})', fontsize=10,
                         color=LEAK_COLORS[code], fontweight='bold')
    axes[idx].legend(fontsize=8, loc='upper right')

axes[-1].set_xlabel('Time [s]')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig15_HilbertEnvelope.png'))
plt.show()
print('✅ Figure 15 saved')

## 7. SECTION F — Statistical Feature Extraction
### Figure 16: Comprehensive Statistical Features — All Leak Types

In [ ]:
def extract_features(signal, fs):
    """Extract 20 statistical + spectral features from a signal."""
    # Time domain
    rms    = np.sqrt(np.mean(signal**2))
    peak   = np.max(np.abs(signal))
    crest  = peak / (rms + 1e-12)
    kurt   = kurtosis(signal)
    skew   = pd.Series(signal).skew()
    std    = np.std(signal)
    energy = np.sum(signal**2)
    shape  = rms / (np.mean(np.abs(signal)) + 1e-12)
    impulse = peak / (np.mean(np.abs(signal)) + 1e-12)
    margin = peak / (np.mean(np.sqrt(np.abs(signal)))**2 + 1e-12)

    # Frequency domain
    freqs, psd = welch(signal, fs=fs, nperseg=1024)
    total_power = np.sum(psd)
    mean_freq  = np.sum(freqs * psd) / (total_power + 1e-12)
    med_freq   = freqs[np.searchsorted(np.cumsum(psd)/total_power, 0.5)]
    spec_var   = np.sum((freqs - mean_freq)**2 * psd) / (total_power + 1e-12)
    spec_std   = np.sqrt(spec_var)
    spec_kurt  = np.sum((freqs - mean_freq)**4 * psd) / ((spec_var**2 + 1e-12) * total_power)

    # Band powers
    bp_low  = np.sum(psd[freqs <= 500])
    bp_mid  = np.sum(psd[(freqs > 500) & (freqs <= 5000)])
    bp_high = np.sum(psd[freqs > 5000])

    # Spectral entropy
    psd_norm = psd / (total_power + 1e-12)
    spec_ent = -np.sum(psd_norm * np.log2(psd_norm + 1e-12))

    return {
        'RMS': rms, 'Peak': peak, 'Crest Factor': crest, 'Kurtosis': kurt,
        'Skewness': skew, 'Std Dev': std, 'Energy': energy,
        'Shape Factor': shape, 'Impulse Factor': impulse, 'Margin Factor': margin,
        'Mean Freq': mean_freq, 'Median Freq': med_freq, 'Spec Std': spec_std,
        'Spec Kurtosis': spec_kurt, 'BP Low': bp_low, 'BP Mid': bp_mid,
        'BP High': bp_high, 'Spec Entropy': spec_ent,
    }


# Extract features for all conditions
feature_records = []
print('Extracting features...')

for topo in TOPOLOGIES:
    for leak in LEAK_TYPES:
        code = LEAK_CODES[leak]
        for flow in FLOW_CONDS:
            for sensor_id in ['A1', 'A2']:
                fpath = find_file(ACC_DIR, topo, leak, flow, sensor_id)
                if fpath is None:
                    continue
                t, v = load_acc_csv(fpath)
                feats = extract_features(v, FS_ACC)
                feats.update({
                    'Topology': topo, 'Leak': code,
                    'Flow': flow, 'Sensor': sensor_id
                })
                feature_records.append(feats)

df_features = pd.DataFrame(feature_records)
print(f'✅ Extracted features: {len(df_features)} records × {len(df_features.columns)} columns')
df_features.head()

### Figure 16: Feature Violin Plots — All Leak Types

In [ ]:
key_features = ['RMS', 'Kurtosis', 'Crest Factor', 'Mean Freq',
                'Spec Entropy', 'BP Low', 'BP Mid', 'BP High']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Feature Distribution per Leak Type — Violin Plots\nAll Topologies & Flow Conditions | Accelerometer',
             fontsize=14, fontweight='bold')

axes_flat = axes.flatten()
palette = {code: color for code, color in LEAK_COLORS.items()}

for idx, feat in enumerate(key_features):
    sns.violinplot(data=df_features, x='Leak', y=feat, palette=palette,
                   ax=axes_flat[idx], inner='box', cut=0)
    axes_flat[idx].set_title(feat, fontsize=11, fontweight='bold')
    axes_flat[idx].set_xlabel('Leak Type')
    axes_flat[idx].set_ylabel(feat)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig16_FeatureViolins.png'))
plt.show()
print('✅ Figure 16 saved')

### Figure 17: Feature Correlation Heatmap

In [ ]:
feat_cols = ['RMS','Peak','Crest Factor','Kurtosis','Skewness','Std Dev',
             'Shape Factor','Impulse Factor','Mean Freq','Median Freq',
             'Spec Std','Spec Kurtosis','BP Low','BP Mid','BP High','Spec Entropy']

fig, ax = plt.subplots(figsize=(14, 11))
corr_mat = df_features[feat_cols].corr()
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask, ax=ax, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', square=True, linewidths=0.5,
            cbar_kws={'label': 'Pearson Correlation'})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig17_FeatureCorrelation.png'))
plt.show()
print('✅ Figure 17 saved')

### Figure 18: Feature Box Plots — Topology Comparison

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Feature Distribution: Branched vs Looped Topology\nAll Leak Types & Flow Conditions',
             fontsize=14, fontweight='bold')

for idx, feat in enumerate(key_features):
    ax = axes[idx//4, idx%4]
    sns.boxplot(data=df_features, x='Topology', y=feat, palette=TOPO_COLORS,
                ax=ax, width=0.5)
    sns.stripplot(data=df_features, x='Topology', y=feat, ax=ax,
                  color='black', size=2, alpha=0.3)
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel('')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig18_Features_TopologyBoxPlot.png'))
plt.show()
print('✅ Figure 18 saved')

## 8. SECTION G — Dimensionality Reduction & Visualization
### Figure 19: PCA Feature Space — All Leak Types

In [ ]:
X = df_features[feat_cols].values
X_scaled = StandardScaler().fit_transform(X)
labels = df_features['Leak'].values
topos  = df_features['Topology'].values

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

fig = plt.figure(figsize=(20, 6))
fig.suptitle('PCA Feature Space Visualization', fontsize=14, fontweight='bold')

# PC1 vs PC2
ax1 = fig.add_subplot(131)
for code in df_features['Leak'].unique():
    mask = labels == code
    ax1.scatter(X_pca[mask, 0], X_pca[mask, 1], label=code,
                color=LEAK_COLORS[code], alpha=0.7, s=50, edgecolors='white', lw=0.3)
ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax1.set_title('PC1 vs PC2 — Leak Type')
ax1.legend()

# PC1 vs PC3
ax2 = fig.add_subplot(132)
for code in df_features['Leak'].unique():
    mask = labels == code
    ax2.scatter(X_pca[mask, 0], X_pca[mask, 2], label=code,
                color=LEAK_COLORS[code], alpha=0.7, s=50, edgecolors='white', lw=0.3)
ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax2.set_ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.1f}%)')
ax2.set_title('PC1 vs PC3 — Leak Type')
ax2.legend()

# Topology
ax3 = fig.add_subplot(133)
for topo in TOPOLOGIES:
    mask = topos == topo
    ax3.scatter(X_pca[mask, 0], X_pca[mask, 1], label=topo,
                color=TOPO_COLORS[topo], alpha=0.6, s=50,
                marker='o' if topo == 'Branched' else '^',
                edgecolors='white', lw=0.3)
ax3.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax3.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax3.set_title('PC1 vs PC2 — Topology')
ax3.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig19_PCA_FeatureSpace.png'))
plt.show()

print(f'\nPCA Explained Variance:')
for i, ev in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {ev*100:.2f}%')
print('✅ Figure 19 saved')

### Figure 20: t-SNE Visualization

In [ ]:
tsne = TSNE(n_components=2, perplexity=min(30, len(X_scaled)-1),
            random_state=42, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('t-SNE Visualization of Extracted Features', fontsize=14, fontweight='bold')

# By leak type
for code in df_features['Leak'].unique():
    mask = labels == code
    axes[0].scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=code,
                    color=LEAK_COLORS[code], alpha=0.75, s=60, edgecolors='white', lw=0.3)
axes[0].set_title('t-SNE by Leak Type')
axes[0].legend()
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')

# By topology
for topo in TOPOLOGIES:
    mask = topos == topo
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=topo,
                    color=TOPO_COLORS[topo], alpha=0.6, s=60,
                    marker='o' if topo == 'Branched' else '^',
                    edgecolors='white', lw=0.3)
axes[1].set_title('t-SNE by Topology')
axes[1].legend()
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')

# By flow condition
flows = df_features['Flow'].values
for flow in FLOW_CONDS:
    mask = flows == flow
    axes[2].scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=flow,
                    color=FLOW_COLORS[flow], alpha=0.65, s=60, edgecolors='white', lw=0.3)
axes[2].set_title('t-SNE by Flow Condition')
axes[2].legend()
axes[2].set_xlabel('t-SNE 1')
axes[2].set_ylabel('t-SNE 2')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig20_TSNE_FeatureSpace.png'))
plt.show()
print('✅ Figure 20 saved')

## 9. SECTION H — Transient Analysis (KEY FOR MSSP)
### Figure 21: Transient Event Detection & Characterization

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(18, 16))
fig.suptitle('Transient Flow Condition — Signal Analysis\nA1 (left) | A2 (right) — All Leak Types | Branched',
             fontsize=14, fontweight='bold')

for idx, leak in enumerate(LEAK_TYPES):
    code = LEAK_CODES[leak]
    result = load_sensor_pair(ACC_DIR, 'Branched', leak, 'Transient', 'A1', 'A2')
    if result is None:
        for jdx in range(2):
            axes[idx, jdx].text(0.5, 0.5, 'File not found',
                                ha='center', transform=axes[idx, jdx].transAxes)
        continue
    t1, sig1, t2, sig2 = result

    for jdx, (sig, sid) in enumerate([(sig1, 'A1'), (sig2, 'A2')]):
        # Full signal
        axes[idx, jdx].plot(t1, sig, color=LEAK_COLORS[code], lw=0.5, alpha=0.7)
        # Highlight transient zone (flow changes at ~20s)
        axes[idx, jdx].axvspan(19.5, 22, color='red', alpha=0.1, label='Transient Zone')
        axes[idx, jdx].axvline(20, color='red', ls='--', lw=1.2, alpha=0.8)
        axes[idx, jdx].set_title(f'{code} — {sid}', fontsize=9,
                                   color=LEAK_COLORS[code], fontweight='bold')
        axes[idx, jdx].set_ylabel('Accel [g]')
        if idx == 0 and jdx == 0:
            axes[idx, jdx].legend(fontsize=8)

for ax in axes[-1]:
    ax.set_xlabel('Time [s]')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig21_Transient_Analysis.png'))
plt.show()
print('✅ Figure 21 saved')

### Figure 22: Pre vs Post Transient Feature Shift

In [ ]:
# Compare features before (0-18s) and after (22-30s) transient event
transient_features = []

for leak in LEAK_TYPES:
    code = LEAK_CODES[leak]
    fpath = find_file(ACC_DIR, 'Branched', leak, 'Transient', 'A1')
    if fpath is None:
        continue
    t, v = load_acc_csv(fpath)

    # Pre-transient
    pre_mask  = t < 18
    post_mask = t > 22
    for segment, label in [(v[pre_mask], 'Pre'), (v[post_mask], 'Post')]:
        if len(segment) < 1000:
            continue
        feats = extract_features(segment, FS_ACC)
        feats['Leak'] = code
        feats['Phase'] = label
        transient_features.append(feats)

df_trans = pd.DataFrame(transient_features)

trans_feats = ['RMS', 'Kurtosis', 'Crest Factor', 'Mean Freq', 'Spec Entropy', 'BP Low']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Feature Shift: Pre-Transient vs Post-Transient\nBranched Network | Sensor A1',
             fontsize=14, fontweight='bold')

for idx, feat in enumerate(trans_feats):
    ax = axes[idx//3, idx%3]
    sns.barplot(data=df_trans, x='Leak', y=feat, hue='Phase',
                palette={'Pre': '#457B9D', 'Post': '#E63946'}, ax=ax,
                capsize=0.1, errwidth=1.5)
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel('Leak Type')
    ax.legend(title='Phase')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig22_TransientFeatureShift.png'))
plt.show()
print('✅ Figure 22 saved')

## 10. SECTION I — Multi-Sensor Cross-Modal Analysis
### Figure 23: Signal Comparison Across All 3 Sensor Types

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(22, 12))
fig.suptitle('Multi-Sensor Signal Comparison — All Leak Types\nBranched | 0.18 LPS (Acc/Pres) | N condition (Hydro)',
             fontsize=14, fontweight='bold')

sensor_configs = [
    ('Accelerometer', ACC_DIR, '0.18 LPS', 'A1', False, FS_ACC),
    ('Dynamic Pressure', DYN_DIR, '0.18 LPS', 'P1', False, FS_ACC),
    ('Hydrophone', HYD_DIR, '0.18 LPS', 'H1', True, FS_HYD),
]
sensor_colors = ['#E63946', '#457B9D', '#2A9D8F']

for row, (sname, sdir, flow, sid, is_raw, fs) in enumerate(sensor_configs):
    for col, leak in enumerate(LEAK_TYPES):
        code = LEAK_CODES[leak]
        fpath = find_file(sdir, 'Branched', leak, flow, sid)
        if fpath is None:
            axes[row, col].text(0.5, 0.5, 'N/A', ha='center', transform=axes[row, col].transAxes)
            continue
        if is_raw:
            t, v = load_hydrophone_raw(fpath, fs=fs)
        else:
            t, v = load_acc_csv(fpath)

        mask = t <= 3
        axes[row, col].plot(t[mask], v[mask], color=sensor_colors[row], lw=0.6)
        if row == 0:
            axes[row, col].set_title(f'{code}', fontsize=10, fontweight='bold',
                                      color=LEAK_COLORS[code])
        if col == 0:
            axes[row, col].set_ylabel(f'{sname}\n[g or Pa or V]', fontsize=9)

for ax in axes[-1]:
    ax.set_xlabel('Time [s]')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig23_MultiSensor_AllLeaks.png'))
plt.show()
print('✅ Figure 23 saved')

### Figure 24: PSD Comparison Across Sensor Types

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('PSD Comparison Across Sensor Types — All Leak Types\nBranched | 0.18 LPS',
             fontsize=14, fontweight='bold')

for col, (sname, sdir, flow, sid, is_raw, fs) in enumerate(sensor_configs):
    for leak in LEAK_TYPES:
        code = LEAK_CODES[leak]
        fpath = find_file(sdir, 'Branched', leak, flow, sid)
        if fpath is None:
            continue
        if is_raw:
            t, v = load_hydrophone_raw(fpath, fs=fs)
        else:
            t, v = load_acc_csv(fpath)

        freqs_psd, psd = welch(v, fs=fs, nperseg=1024)
        axes[col].semilogy(freqs_psd, psd, label=code,
                            color=LEAK_COLORS[code], lw=1.3, alpha=0.85)

    axes[col].set_title(sname, fontsize=11, fontweight='bold', color=sensor_colors[col])
    axes[col].set_xlabel('Frequency [Hz]')
    axes[col].set_ylabel('PSD')
    axes[col].legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig24_PSD_MultiSensor.png'))
plt.show()
print('✅ Figure 24 saved')

## 11. SECTION J — Wave Propagation Speed Estimation
### Figure 25: Apparent Wave Speed from TDOA — Topology Comparison

In [ ]:
# PVC pipe wave speed typically ~350-420 m/s for structure-borne
# Fluid-borne ~1000-1200 m/s
# Sensor separation assumed (from testbed: 47m total, sensors roughly at 1/4 and 3/4)
SENSOR_SEPARATION = 23.5  # meters (estimated — adjust if you have exact positions)

wave_speed_data = []
for topo in TOPOLOGIES:
    for leak in LEAK_TYPES:
        code = LEAK_CODES[leak]
        for flow in FLOW_CONDS:
            result = load_sensor_pair(ACC_DIR, topo, leak, flow, 'A1', 'A2')
            if result is None:
                continue
            t1, sig1, t2, sig2 = result
            n5 = int(5 * FS_ACC)
            lags, cc = compute_gcc_phat(sig1[:n5], sig2[:n5], FS_ACC)
            mask = np.abs(lags) <= 0.1
            peak_lag = lags[mask][np.argmax(np.abs(cc[mask]))]
            if abs(peak_lag) > 1e-6:
                speed = SENSOR_SEPARATION / abs(peak_lag)
            else:
                speed = np.nan
            wave_speed_data.append({
                'Topology': topo, 'Leak': code, 'Flow': flow,
                'TDOA_ms': peak_lag*1000, 'Wave_Speed_ms': speed
            })

df_wavespeed = pd.DataFrame(wave_speed_data)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Apparent Wave Propagation Speed (A1→A2, separation={SENSOR_SEPARATION}m)\n'
             f'Note: Adjust SENSOR_SEPARATION to actual testbed value',
             fontsize=13, fontweight='bold')

# Box plot by leak type
sns.boxplot(data=df_wavespeed, x='Leak', y='Wave_Speed_ms', hue='Topology',
            palette=TOPO_COLORS, ax=axes[0])
axes[0].set_xlabel('Leak Type')
axes[0].set_ylabel('Wave Speed [m/s]')
axes[0].set_title('Wave Speed per Leak Type')
axes[0].axhline(420, color='green', ls='--', lw=1.5, label='Typical PVC structure-borne')
axes[0].axhline(1200, color='blue', ls='--', lw=1.5, label='Typical fluid-borne')
axes[0].legend(fontsize=8)

# Box plot by flow condition
sns.boxplot(data=df_wavespeed, x='Flow', y='Wave_Speed_ms', hue='Topology',
            palette=TOPO_COLORS, ax=axes[1])
axes[1].set_xlabel('Flow Condition')
axes[1].set_ylabel('Wave Speed [m/s]')
axes[1].set_title('Wave Speed per Flow Condition')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'Fig25_WaveSpeed.png'))
plt.show()
print('✅ Figure 25 saved')
print('\nWave Speed Summary:')
print(df_wavespeed.groupby(['Topology','Leak'])['Wave_Speed_ms'].describe().round(1))

## 12. SECTION K — Summary Dashboard
### Figure 26: Complete Analysis Summary

In [ ]:
fig = plt.figure(figsize=(22, 16))
fig.suptitle('Comprehensive Analysis Summary Dashboard\n'
             'WDS Leak Characterization — Aghashahi et al. (2022) Dataset',
             fontsize=15, fontweight='bold', y=0.98)

gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# Panel 1: RMS by leak × topology
ax1 = fig.add_subplot(gs[0, 0:2])
sns.barplot(data=df_features, x='Leak', y='RMS', hue='Topology',
            palette=TOPO_COLORS, ax=ax1, capsize=0.1)
ax1.set_title('RMS Energy by Leak Type & Topology', fontweight='bold')
ax1.set_xlabel('Leak Type'); ax1.set_ylabel('RMS [g]')

# Panel 2: Kurtosis by flow
ax2 = fig.add_subplot(gs[0, 2:4])
sns.barplot(data=df_features, x='Leak', y='Kurtosis', hue='Flow',
            palette=FLOW_COLORS, ax=ax2, capsize=0.1)
ax2.set_title('Kurtosis by Leak Type & Flow Condition', fontweight='bold')
ax2.set_xlabel('Leak Type'); ax2.set_ylabel('Kurtosis')

# Panel 3: TDOA summary
ax3 = fig.add_subplot(gs[1, 0:2])
df_tdoa_plot = df_wavespeed.pivot_table(index='Leak', columns='Topology',
                                         values='TDOA_ms', aggfunc='mean')
df_tdoa_plot.plot(kind='bar', ax=ax3, color=[TOPO_COLORS[t] for t in df_tdoa_plot.columns],
                  edgecolor='white', width=0.7)
ax3.set_title('Mean TDOA (A1-A2) by Leak Type', fontweight='bold')
ax3.set_xlabel('Leak Type'); ax3.set_ylabel('TDOA [ms]')
ax3.axhline(0, color='black', lw=0.8, ls='--')
ax3.tick_params(axis='x', rotation=30)

# Panel 4: Spectral entropy
ax4 = fig.add_subplot(gs[1, 2:4])
sns.violinplot(data=df_features, x='Leak', y='Spec Entropy',
               palette=LEAK_COLORS, ax=ax4, inner='box', cut=0)
ax4.set_title('Spectral Entropy Distribution', fontweight='bold')
ax4.set_xlabel('Leak Type'); ax4.set_ylabel('Spectral Entropy')

# Panel 5: Band energy radar
ax5 = fig.add_subplot(gs[2, 0:2])
df_band_mean = df_features.groupby('Leak')[['BP Low', 'BP Mid', 'BP High']].mean()
df_band_norm = df_band_mean.div(df_band_mean.sum(axis=1), axis=0) * 100
df_band_norm.T.plot(kind='bar', ax=ax5,
                    color=[LEAK_COLORS[c] for c in df_band_norm.index],
                    edgecolor='white', width=0.8)
ax5.set_title('Band Energy Distribution (%)', fontweight='bold')
ax5.set_xlabel('Frequency Band'); ax5.set_ylabel('Energy %')
ax5.set_xticklabels(['Low (<500Hz)', 'Mid (500-5kHz)', 'High (>5kHz)'], rotation=15)

# Panel 6: PCA explained variance
ax6 = fig.add_subplot(gs[2, 2:4])
pca_full = PCA().fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100
ax6.bar(range(1, len(cumvar)+1), pca_full.explained_variance_ratio_*100,
        color='#457B9D', alpha=0.8, label='Individual')
ax6_twin = ax6.twinx()
ax6_twin.plot(range(1, len(cumvar)+1), cumvar, 'r-o', ms=4, lw=1.5, label='Cumulative')
ax6_twin.axhline(95, color='red', ls='--', lw=1, alpha=0.5)
ax6.set_xlabel('Principal Component')
ax6.set_ylabel('Explained Variance [%]')
ax6_twin.set_ylabel('Cumulative Variance [%]', color='red')
ax6.set_title('PCA Explained Variance', fontweight='bold')
ax6.set_xlim(0, 15)

plt.savefig(os.path.join(OUT_DIR, 'Fig26_Summary_Dashboard.png'))
plt.show()
print('✅ Figure 26 saved')

## 13. Save Feature Dataset

In [ ]:
# Save extracted features
feat_csv = os.path.join(OUT_DIR, '..', 'extracted_features.csv')
df_features.to_csv(feat_csv, index=False)
print(f'✅ Features saved: {feat_csv}')
print(f'   Shape: {df_features.shape}')

# Save TDOA results
tdoa_csv = os.path.join(OUT_DIR, '..', 'tdoa_wavespeed.csv')
df_wavespeed.to_csv(tdoa_csv, index=False)
print(f'✅ TDOA/WaveSpeed saved: {tdoa_csv}')

print('\n' + '='*60)
print('ALL ANALYSIS COMPLETE')
print('='*60)
print(f'Figures saved to: {OUT_DIR}')
print(f'Total figures generated: 26+')
print('\nFigure Index:')
figs = [
    'Fig01 — Time domain: all leak types',
    'Fig02 — Sensor pair A1 vs A2',
    'Fig03 — Flow condition effect',
    'Fig04 — Topology comparison BR vs LO',
    'Fig05 — PSD all leak types',
    'Fig06 — FFT leak vs no-leak',
    'Fig07 — Band energy distribution',
    'Fig08 — STFT spectrograms',
    'Fig09 — CWT scalograms',
    'Fig10 — Wavelet packet decomposition',
    'Fig11 — GCC-PHAT cross-correlation TDOA',
    'Fig12 — TDOA heatmap all conditions',
    'Fig13 — Coherence A1-A2',
    'Fig14 — Topology coherence comparison',
    'Fig15 — Hilbert envelope burst analysis',
    'Fig16 — Feature violin plots',
    'Fig17 — Feature correlation heatmap',
    'Fig18 — Feature topology boxplots',
    'Fig19 — PCA feature space',
    'Fig20 — t-SNE visualization',
    'Fig21 — Transient event analysis',
    'Fig22 — Pre vs post transient features',
    'Fig23 — Multi-sensor signal comparison',
    'Fig24 — PSD multi-sensor comparison',
    'Fig25 — Wave propagation speed',
    'Fig26 — Summary dashboard',
]
for f in figs:
    print(f'  {f}')